# V38 — Model-Free Evidence 2: Household Switching and Variance Decomposition

**Purpose.** Update the original four-category model-free analysis to the **V38 five-category specification**, with ordinary milk incorporated into healthy beverages. The notebook produces (i) an exact between/within-household decomposition of annual healthy food-spending-share variation, (ii) a 2008–2017 chart of annual cross-sectional variance, and (iii) three purposively selected household trajectories with repeated descriptive regime crossings.

**Annual outcome:** For each household-year, let

\[
Y_{it}=\frac{\sum_{j\in(i,t)}\left(\text{fresh FV} + \text{fat-free yogurt} + \text{low-sugar cereal} + \text{healthy snack} + \text{healthy beverage} + \text{all ordinary milk spending}\right)_{ijt}}{\sum_{j\in(i,t)}\texttt{food\_spend}_{ijt}}.
\]

The five V38 categories are **FV, yogurt, cereal, snacks, and beverages (including milk)**. Ordinary milk contributes the **skimmed, low-fat, and other milk** columns to the healthy-beverage numerator. The merge of *trip-level choice codes* matches V38 (`milk_y > 0 OR beverage_y == 1` gives healthy beverage code 1, otherwise original beverage code 2 or 0), but this model-free outcome is **a spending share**, not the HMM's gated incidence/choice outcome or a within-category share. Milk is not counted twice. Sweetened/flavored products are **not newly reclassified**: only the milk already represented in V38's `milk_y` and milk spending columns counts.

**V38 sample.** Use the existing 2008–2017 trip-level HMM Parquet. V38 sets `MIN_OBSERVED_YEARS_PER_HOUSEHOLD=1`, so this notebook imposes **no additional household tenure filter**. The variance sample still requires a valid household-year with at least 12 trips (inherited descriptive screen). The repeated-switch illustrations continue to require at least 8 consecutive eligible years, 3 clear regime crossings, and a pooled median ±2.5 percentage-point neutral band. This is *purposive illustration*, not a population estimate of transition probabilities or estimated HMM states.

**Decomposition.** All eligible household-year observations enter the exact decomposition, including unbalanced panels and short spells. A separate sensitivity analysis selects households with at least one ≥8-year consecutive eligible spell. Year-by-year cross-sectional variance is reported separately from between-household variation.

**Privacy and reproducibility.** The original restricted Parquet is read-only. All Parquet, detailed household files, aggregate CSVs and figures are stored under `healthy_purchase_data/model_outputs/model_free_evidence/v38_plain_milk_beverage/switching_and_variance/`, **outside the GitHub repository**. Review any figures/tables for sharing authorization before uploading. Set `HEALTHY_PURCHASE_DATA_ROOT` to change local data location. Run cells in order using Python, DuckDB, NumPy, Pandas, PyArrow, and Matplotlib.


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D

# Keep all restricted data, caches and household-specific figures outside the Git repository.
DATA_ROOT = Path(os.environ.get(
    "HEALTHY_PURCHASE_DATA_ROOT", r"C:\Users\rwen13\Documents\healthy_purchase_data"
))
SOURCE = DATA_ROOT / "processed" / "hmm_input" / "hmm_estimation_panel.parquet"
OUTPUT_DIR = (DATA_ROOT / "model_outputs" / "model_free_evidence"
              / "v38_plain_milk_beverage" / "switching_and_variance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START, YEAR_END = 2008, 2017
MIN_ANNUAL_TRIPS = 12   # Retained from original model-free variance/switching analysis.
N_EXAMPLES = 3
MIN_CONSECUTIVE_YEARS = 8
MIN_SWITCHES = 3
EXCLUSION_BAND_PP = 2.5
RANDOM_SEED = 20261031
DUCKDB_THREADS = min(4, max(1, (os.cpu_count() or 2) // 2))
DUCKDB_MEMORY_LIMIT = "2GB"
CHECK_DUPLICATE_TRIPS = True
AGGREGATION_VERSION = "v38-five-category-milk-as-beverage-v1"

# Use the same eight mutually exclusive healthy spending components as V38 DID.
# Milk has no separate category: every ordinary-milk dollar is included in beverages.
HEALTHY_COMPONENTS = {
    "fv": ["fresh_spending"],
    "yogurt": ["skimmed_yogurt_spending"],
    "cereal": ["low_sugar_cereal_spending"],
    "snack": ["healthy_snack_spending"],
    "beverage": [
        "healthy_beverage_spending", "skimmed_milk_spending",
        "low_fat_milk_spending", "other_milk_spending",
    ],
}
HEALTHY_SPENDING_COLUMNS = [col for cols in HEALTHY_COMPONENTS.values() for col in cols]
ORIGINAL_OUTCOME_COLUMNS = ["fv_y", "yogurt_y", "cereal_y", "snack_y", "beverage_y", "milk_y"]
CACHE = OUTPUT_DIR / "annual_v38_five_category_spending.parquet"
CACHE_META = OUTPUT_DIR / "annual_v38_five_category_spending.meta.json"
assert len(HEALTHY_COMPONENTS) == 5 and len(HEALTHY_SPENDING_COLUMNS) == 8
assert MIN_SWITCHES >= 3 and EXCLUSION_BAND_PP > 0
print("V38 source:", SOURCE)
print("Healthy components:", HEALTHY_COMPONENTS)


## 1. Validate V38 trip data and aggregate to the household-year level

Use DuckDB with a bounded memory setting, scanning the same 38-column Parquet as V38 and the V38 matched-DID notebook. Validate positive/finite denominators, amounts, valid original milk/beverage codes, purchase-date years, and unique trip keys. The source fields are **not relabeled or overwritten**. A separate nine-combination audit documents exactly how original milk/beverage choice codes map to V38 merged beverage codes. An annual cache is reused only when source size/mtime and the specification agree. `food_spend` always remains the original annual total food spending, not a five-category spending sum.


In [ ]:
def qi(column):
    """Quote trusted DuckDB identifiers, preserving original column spelling."""
    return '"' + column.replace('"', '""') + '"'


if not SOURCE.is_file():
    raise FileNotFoundError(f"V38 input HMM trip panel not found: {SOURCE}")

cache_metadata = {
    "aggregation_version": AGGREGATION_VERSION,
    "source": str(SOURCE.resolve()),
    "size": SOURCE.stat().st_size,
    "mtime_ns": SOURCE.stat().st_mtime_ns,
    "healthy_components": HEALTHY_COMPONENTS,
    "outcome_columns": ORIGINAL_OUTCOME_COLUMNS,
    "years": [YEAR_START, YEAR_END],
    "check_duplicate_trips": CHECK_DUPLICATE_TRIPS,
}

if (CACHE.is_file() and CACHE_META.is_file()
        and json.loads(CACHE_META.read_text(encoding="utf-8")) == cache_metadata):
    annual = pd.read_parquet(CACHE)
    print(f"Loaded V38 annual cache: {len(annual):,} household-years")
else:
    con = duckdb.connect()
    try:
        con.execute(f"SET threads={DUCKDB_THREADS}")
        con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        temp_dir = OUTPUT_DIR / "duckdb_temp"
        temp_dir.mkdir(parents=True, exist_ok=True)
        escaped_temp = temp_dir.as_posix().replace("'", "''")
        con.execute(f"SET temp_directory='{escaped_temp}'")
        escaped_source = SOURCE.as_posix().replace("'", "''")
        con.execute(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{escaped_source}')")
        available = {r[0] for r in con.execute("DESCRIBE trips").fetchall()}
        required = ({"hhid", "panel_year", "trip_id", "purchase_date", "food_spend"}
                    | set(ORIGINAL_OUTCOME_COLUMNS) | set(HEALTHY_SPENDING_COLUMNS))
        missing = sorted(required - available)
        if missing:
            raise ValueError(f"Missing required V38 panel columns: {missing}")

        # Match V38 and the V38 DID input validation; null category codes are errors.
        invalid_predicates = [
            "hhid IS NULL", "panel_year IS NULL", "trip_id IS NULL", "purchase_date IS NULL",
            f"panel_year NOT BETWEEN {YEAR_START} AND {YEAR_END}",
            "TRY_CAST(purchase_date AS DATE) IS NULL",
            "YEAR(TRY_CAST(purchase_date AS DATE)) != panel_year",
        ]
        money_columns = ["food_spend", *HEALTHY_SPENDING_COLUMNS]
        for col in money_columns:
            invalid_predicates.append(
                f"({qi(col)} IS NULL OR NOT isfinite(TRY_CAST({qi(col)} AS DOUBLE)) "
                f"OR {qi(col)} < 0)"
            )
        for col in ORIGINAL_OUTCOME_COLUMNS:
            allowed = "(0,1,2,3)" if col == "cereal_y" else "(0,1,2)"
            invalid_predicates.append(
                f"({qi(col)} IS NULL OR NOT isfinite(TRY_CAST({qi(col)} AS DOUBLE)) "
                f"OR {qi(col)} NOT IN {allowed})"
            )
        invalid_where = " OR ".join(f"({p})" for p in invalid_predicates)
        n_rows, n_invalid = con.execute(
            f"SELECT count(*), count(*) FILTER (WHERE {invalid_where}) FROM trips"
        ).fetchone()
        if n_rows == 0 or n_invalid:
            raise ValueError(
                f"V38 trip audit failed: {n_rows:,} trips and {n_invalid:,} invalid rows"
            )
        if CHECK_DUPLICATE_TRIPS:
            duplicate = con.execute("""
                SELECT 1 FROM trips GROUP BY hhid, panel_year, trip_id
                HAVING count(*) > 1 LIMIT 1
            """).fetchone()
            if duplicate:
                raise ValueError("Duplicate household-year-trip keys detected in V38 input")

        # V38 priority rule: any original milk purchase is a healthy beverage choice.
        # This is a choice-code audit only; the annual Y below uses actual dollar amounts.
        merge_audit = con.execute("""
            SELECT milk_y AS original_milk_y, beverage_y AS original_beverage_y,
                   CASE WHEN milk_y IN (1,2) OR beverage_y = 1 THEN 1
                        WHEN beverage_y = 2 THEN 2 ELSE 0 END AS merged_beverage_y,
                   count(*) AS n_trips
            FROM trips
            GROUP BY 1, 2, 3 ORDER BY 1, 2
        """).df()
        merge_audit.to_csv(OUTPUT_DIR / "v38_beverage_merge_audit.csv", index=False)
        if merge_audit.n_trips.sum() != n_rows:
            raise AssertionError("V38 beverage merge audit does not sum to all trips")

        # All eight columns have already passed finite/nonnegative validation.
        # Keep spending components separate in the source: milk is included only once.
        healthy_expr = " + ".join(qi(col) for col in HEALTHY_SPENDING_COLUMNS)
        annual = con.execute(f"""
            SELECT CAST(hhid AS VARCHAR) AS hhid,
                   CAST(panel_year AS INTEGER) AS panel_year,
                   count(*) AS n_trips,
                   sum(CAST(food_spend AS DOUBLE)) AS food_spend,
                   sum({healthy_expr}) AS healthy_spend
            FROM trips
            GROUP BY hhid, panel_year
        """).df()
        audit = pd.DataFrame([
            {"metric": "input_trips", "value": int(n_rows)},
            {"metric": "invalid_source_rows", "value": int(n_invalid)},
            {"metric": "household_years", "value": len(annual)},
            {"metric": "households", "value": annual.hhid.nunique()},
            {"metric": "original_milk_purchase_trips",
             "value": int(merge_audit.loc[merge_audit.original_milk_y.gt(0), "n_trips"].sum())},
            {"metric": "merged_beverage_purchase_trips",
             "value": int(merge_audit.loc[merge_audit.merged_beverage_y.gt(0), "n_trips"].sum())},
            {"metric": "merged_beverage_healthy_choice_trips",
             "value": int(merge_audit.loc[merge_audit.merged_beverage_y.eq(1), "n_trips"].sum())},
        ])
        audit.to_csv(OUTPUT_DIR / "input_audit.csv", index=False)
    finally:
        con.close()

    if annual.duplicated(["hhid", "panel_year"]).any():
        raise ValueError("Aggregation produced duplicate household-year keys")
    if annual.healthy_spend.lt(0).any() or annual.food_spend.lt(0).any():
        raise ValueError("Negative annual spending detected")
    annual["Y"] = annual.healthy_spend.div(annual.food_spend.where(annual.food_spend > 0))
    out_of_range = annual.Y.notna() & ~annual.Y.between(-1e-9, 1 + 1e-9)
    if out_of_range.any():
        raise ValueError(
            f"Healthy spending / food spending outside [0,1] in "
            f"{int(out_of_range.sum()):,} household-years: check source component overlap"
        )
    if annual.Y.notna().sum() == 0:
        raise ValueError("No positive-food-spending household-years")
    annual = annual.sort_values(["hhid", "panel_year"]).reset_index(drop=True)
    annual.to_parquet(CACHE, index=False)
    CACHE_META.write_text(json.dumps(cache_metadata, indent=2), encoding="utf-8")
    print(f"Aggregated {n_rows:,} trips into {len(annual):,} household-years")

if annual.duplicated(["hhid", "panel_year"]).any():
    raise ValueError("Cached V38 annual data contain duplicate household-year keys")
print(f"Households: {annual.hhid.nunique():,}; years: {annual.panel_year.min()}–{annual.panel_year.max()}")


## 2. Define the eligible descriptive sample (the original 12-trip threshold)

A valid household-year must have at least `MIN_ANNUAL_TRIPS = 12` trips, a strictly positive annual food-spending denominator, and a share between 0 and 1. One- or two-year households are **not excluded by the V38 HMM input rule**: every valid observed year contributes to the main variance decomposition. A consecutive-year spell is broken at any missing/invalid calendar year; the three illustrative households and long-panel robustness require a qualifying ≥8-year spell. The yearly sample-flow audit makes these restrictions visible.


In [ ]:
valid = annual.loc[
    annual.Y.notna() & annual.Y.between(0, 1)
    & (annual.n_trips >= MIN_ANNUAL_TRIPS),
    ["hhid", "panel_year", "n_trips", "Y"]
].sort_values(["hhid", "panel_year"]).copy()

if valid.empty:
    raise ValueError("No household-years satisfy the annual validity criteria")

# A missing or invalid year starts a new spell; it is never bridged.
new_spell = valid.hhid.ne(valid.hhid.shift()) | valid.panel_year.diff().ne(1)
valid["spell_id"] = new_spell.cumsum().astype("int64")
print(f"Variance sample: {len(valid):,} household-years, {valid.hhid.nunique():,} households")

# Main variance sample (not a tenure-balanced panel): all eligible observed years.
sample_flow = pd.DataFrame([
    {"stage": "Input household-years", "household_years": len(annual),
     "households": annual.hhid.nunique()},
    {"stage": "Valid share 0-1, positive food spending, and annual trips >= 12",
     "household_years": len(valid), "households": valid.hhid.nunique()},
])
sample_flow.to_csv(OUTPUT_DIR / "variance_sample_flow.csv", index=False)
display(sample_flow)


## 3. Exact between/within household variance decomposition

For annual healthier-food-spending share \(Y_{it}\), let \(T_i\) be the number of eligible years for household \(i\), \(\bar Y_i\) its eligible-year mean, and \(\bar Y\) the household-year observation-weighted grand mean. The exact unbalanced-panel identity is:

\[
\sum_{it}(Y_{it}-\bar Y)^2
=\sum_i T_i(\bar Y_i-\bar Y)^2
+\sum_{it}(Y_{it}-\bar Y_i)^2.
\]

The **between and within shares** use the corresponding sums of squares divided by total sum of squares. Every eligible household-year is equally weighted, with no imputation of missing years and no HMM posteriors. The reported `variance` components divide their sums of squares by the total household-year count and are in *share-squared units*: to express a variance in squared percentage points (pp²), multiply by **10,000**. **For example, 0.01 in share-squared units corresponds to 100 pp²**, not 1 pp². The within component reflects both genuine behavior changes and measurement noise, not verified latent-state switches.


In [ ]:
def decompose_between_within(frame: pd.DataFrame) -> pd.DataFrame:
    """Exactly partition unweighted household-year outcome variation.

    Each observed household-year contributes equally. Household means are based only on
    eligible observed years, so an unbalanced panel is handled without imputation.
    """
    if frame.empty or frame.Y.isna().any():
        raise ValueError("Decomposition input must be nonempty with no missing Y")
    household_mean = frame.groupby("hhid")["Y"].transform("mean")
    overall_mean = float(frame.Y.mean())
    between_ss = float(np.square(household_mean - overall_mean).sum())
    within_ss = float(np.square(frame.Y - household_mean).sum())
    total_ss = float(np.square(frame.Y - overall_mean).sum())
    if not np.isclose(between_ss + within_ss, total_ss, rtol=1e-10, atol=1e-12):
        raise AssertionError("Between + within must equal total sum of squares")
    if total_ss <= 0:
        raise ValueError("Zero total variation: percentage decomposition is undefined")
    n_obs = len(frame)
    return pd.DataFrame([
        {"component": "Between households", "sum_of_squares": between_ss,
         "variance": between_ss / n_obs, "share_percent": 100 * between_ss / total_ss},
        {"component": "Within households across years", "sum_of_squares": within_ss,
         "variance": within_ss / n_obs, "share_percent": 100 * within_ss / total_ss},
        {"component": "Total", "sum_of_squares": total_ss,
         "variance": total_ss / n_obs, "share_percent": 100.0},
    ]).assign(n_household_years=n_obs, n_households=frame.hhid.nunique(),
             overall_mean_share=overall_mean)

variance_decomposition = decompose_between_within(valid)
variance_decomposition.to_csv(OUTPUT_DIR / "variance_decomposition_v38_five_category.csv", index=False)
display(variance_decomposition.round(6))

# Extra sensitivity check: restrict to households with a consecutive >=8-year valid spell.
spell_length = valid.groupby("spell_id").size()
long_spell_ids = spell_length.index[spell_length >= MIN_CONSECUTIVE_YEARS]
robust_households = valid.loc[valid.spell_id.isin(long_spell_ids), "hhid"].unique()
robust_valid = valid.loc[valid.hhid.isin(robust_households)].copy()
if not robust_valid.empty:
    assert robust_valid.hhid.nunique() == len(robust_households)
    variance_robustness = decompose_between_within(robust_valid)
    variance_robustness.to_csv(OUTPUT_DIR / "variance_decomposition_v38_long_panel_robustness.csv", index=False)
    print("Robustness: households having at least one >=8-year consecutive valid spell; all their valid years included")
    display(variance_robustness.round(6))
else:
    print("No households meet the >=8 consecutive eligible years robustness condition")


## 4. Calendar-year cross-sectional variation (a distinct quantity)

Compute the mean and variance **across households within each calendar year**, including both population (`ddof=0`) and sample (`ddof=1`) variance. Report the average of the ten year-specific population variances as a separate descriptive statistic. This is not the within-household component of the between/within decomposition and cannot be substituted for it.


In [ ]:
yearly_cross_section = (
    valid.groupby("panel_year", sort=True)
    .agg(n_households=("hhid", "nunique"),
         mean_share=("Y", "mean"),
         variance_population=("Y", lambda s: float(s.var(ddof=0))),
         variance_sample=("Y", lambda s: float(s.var(ddof=1)) if len(s) > 1 else np.nan),
         std_population=("Y", lambda s: float(s.std(ddof=0))))
    .reset_index()
)
yearly_cross_section["mean_share_percent"] = yearly_cross_section.mean_share * 100
yearly_cross_section["std_percentage_points"] = yearly_cross_section.std_population * 100
# Convert from share-squared units to (percentage points)^2 for interpretation.
yearly_cross_section["variance_pp_squared"] = yearly_cross_section.variance_population * 10000
yearly_cross_section.to_csv(OUTPUT_DIR / "yearly_cross_sectional_variance_v38_five_category.csv", index=False)
display(yearly_cross_section.round(5))
print("Mean annual cross-sectional population variance (share-squared):",
      round(float(yearly_cross_section.variance_population.mean()), 6))


## 5. Figures A and B: between/within shares and yearly cross-sectional variance

**Figure A:** The exact between-household and within-household percentages of **total annual household-year variation**, displayed in one 100%-stacked horizontal bar. **Figure B:** The year-specific cross-household variance of annual healthy shares in squared percentage points (pp²). These are conceptually different statistics. Both figures use the V38 five-category spending definition with ordinary milk in beverages.


In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
BLUE, ORANGE = "#2166AC", "#E58B45"
shares = variance_decomposition.set_index("component").loc[
    ["Between households", "Within households across years"], "share_percent"
].to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(11.5, 3.2))
left = 0.0
for fraction, label, color in zip(shares, ["Between households", "Within households"], [BLUE, ORANGE]):
    ax.barh(0, fraction, left=left, height=0.52, color=color, edgecolor="white", linewidth=2, label=label)
    if fraction >= 8:
        ax.text(left + fraction / 2, 0, f"{fraction:.1f}%", color="white", ha="center",
                va="center", fontsize=17, fontweight="bold")
    else:
        ax.annotate(f"{fraction:.1f}%", (left + fraction / 2, 0), xytext=(0, 20),
                    textcoords="offset points", ha="center", fontsize=10, color=color)
    left += fraction
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xticks([0, 20, 40, 60, 80, 100], ["0%", "20%", "40%", "60%", "80%", "100%"])
ax.grid(axis="x", alpha=0.12)
ax.set_axisbelow(True)
ax.set_title("V38 Annual Healthy Purchasing: Variance Decomposition", fontsize=17, fontweight="bold", pad=15)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
fig.text(.06, .03,
         f"V38 five categories (milk in beverages) | {len(valid):,} household-years | {valid.hhid.nunique():,} households | "
         f"{YEAR_START}–{YEAR_END} | Equal weight per household-year", fontsize=9, color="#555555")
fig.subplots_adjust(left=.06, right=.98, top=.75, bottom=.32)
for ext in ("png", "pdf"):
    fig.savefig(OUTPUT_DIR / f"variance_decomposition_v38_five_category.{ext}", dpi=300, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4.5))
ax2.bar(yearly_cross_section.panel_year.astype(str), yearly_cross_section.variance_pp_squared,
        color=BLUE, alpha=0.85)
ax2.set_xlabel("Calendar year")
ax2.set_ylabel("Variance of annual healthy share (pp²)")
ax2.set_title("Cross-Sectional Variance Across Households Within Each Year", fontsize=14,
              fontweight="bold", pad=14)
ax2.grid(axis="y", alpha=.18)
ax2.set_axisbelow(True)
fig2.tight_layout()
for ext in ("png", "pdf"):
    fig2.savefig(OUTPUT_DIR / f"annual_cross_sectional_variance_v38_five_category.{ext}",
                 dpi=300, bbox_inches="tight")
plt.show()


## 6. Select three households with repeated clear threshold crossings

Preserve the original **purposive selection**: compute the pooled median of eligible household-years, classify points above the median +2.5 pp and below the median -2.5 pp as the two clear descriptive regimes, and treat points inside the band as neutral. Count crossings between *successive nonneutral observations* within an uninterrupted valid-calendar-year spell. For each household, retain only its longest valid spell; require that spell to have ≥8 years and ≥3 crossings. Sort eligible households by crossings, annual range, spell length and a seeded tie break to choose **exactly three**. Neither the pooled median nor these labels represent estimated V38 HMM state probabilities.


In [ ]:
threshold = float(valid.Y.median())
band = EXCLUSION_BAND_PP / 100.0
if not (0 < threshold - band < threshold + band < 1):
    raise ValueError("Exclusion band extends beyond valid share range")

valid["regime"] = np.select(
    [valid.Y >= threshold + band, valid.Y <= threshold - band],
    [1, -1], default=0
).astype("int8")

def count_switches(g):
    """Count clear-regime crossings within an uninterrupted household-year spell."""
    clear = g.regime.to_numpy()
    clear = clear[clear != 0]
    return int(np.sum(clear[1:] != clear[:-1])) if len(clear) > 1 else 0

spell_stats = valid.groupby("spell_id", sort=False).agg(
    hhid=("hhid", "first"), n_years=("panel_year", "size"),
    first_year=("panel_year", "min"), last_year=("panel_year", "max"),
    annual_range_pp=("Y", lambda x: float((x.max()-x.min())*100)),
)
switch_counts = valid.groupby("spell_id", sort=False)["regime"].apply(
    lambda s: int(np.sum(s.to_numpy()[s.to_numpy()!=0][1:] !=
                         s.to_numpy()[s.to_numpy()!=0][:-1]))
)
spell_stats["switches"] = switch_counts.astype(int)
longest = (spell_stats.sort_values(["n_years", "first_year"], ascending=[False, True])
           .drop_duplicates("hhid"))
eligible = longest.loc[
    (longest.n_years >= MIN_CONSECUTIVE_YEARS)
    & (longest.switches >= MIN_SWITCHES)
].copy()
screening_summary = pd.DataFrame([
    {"stage": "All households", "households": annual.hhid.nunique()},
    {"stage": f"At least one valid year (>= {MIN_ANNUAL_TRIPS} trips)", "households": valid.hhid.nunique()},
    {"stage": f"Longest consecutive spell >= {MIN_CONSECUTIVE_YEARS} years", "households": int((longest.n_years >= MIN_CONSECUTIVE_YEARS).sum())},
    {"stage": f"Also >= {MIN_SWITCHES} clear crossings", "households": len(eligible)},
])
screening_summary.to_csv(OUTPUT_DIR / "screening_counts_v38.csv", index=False)
display(screening_summary)
print(f"Pooled median: {threshold*100:.2f}%; exclusion band: ±{EXCLUSION_BAND_PP:g} pp")
if len(eligible) < N_EXAMPLES:
    raise ValueError(f"Only {len(eligible)} illustrative households meet the rules; "
                     f"{N_EXAMPLES} required. Variance outputs above have been saved.")

# Intentional, nonrepresentative examples: maximize switches, range, and spell length.
eligible = eligible.sort_values("hhid").copy()
eligible["tie_break"] = np.random.default_rng(RANDOM_SEED).random(len(eligible))
selected = eligible.sort_values(
    ["switches", "annual_range_pp", "n_years", "tie_break"],
    ascending=[False, False, False, True]
).head(N_EXAMPLES).copy()
selected["example_label"] = [f"Household {chr(65+i)}" for i in range(N_EXAMPLES)]
examples = valid.loc[valid.spell_id.isin(selected.index)].copy()
examples["example_label"] = examples.spell_id.map(selected.example_label)
selected.to_csv(OUTPUT_DIR / "selected_households_v38_private.csv", index_label="spell_id")
examples.to_parquet(OUTPUT_DIR / "selected_trajectories_v38_private.parquet", index=False)
display(selected[["example_label", "n_years", "first_year", "last_year", "switches", "annual_range_pp"]])


## 7. Figure C: three household trajectories (illustrative, not representative)

Retain the original three-panel visual format. The pooled 2008–2017 median is a single descriptive reference line. Blue and red points identify observations outside the neutral band, not HMM inferred states. The figure has shared axes and displays the actual annual V38 healthy-share values. Household IDs stay in private research files; display labels A, B and C are nonidentifying placeholders.


In [ ]:
plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
blue, red, gray = "#2266A5", "#CF3636", "#777777"
fig, axes = plt.subplots(1, 3, figsize=(15, 5.6), sharex=True, sharey=True)

# Shared scale: preserve zero as a reference and the same height in each panel.
max_display = min(100, max(50, int(np.ceil(examples.Y.max()*100/10))*10))
for ax, (_, item) in zip(axes.flat, selected.iterrows()):
    s = examples.loc[examples.example_label.eq(item.example_label)].sort_values("panel_year")
    x, y = s.panel_year.to_numpy(), 100*s.Y.to_numpy()
    ax.axhspan(0, threshold*100, color="#FBEAEA", alpha=0.85, zorder=0)
    ax.axhspan(threshold*100, max_display, color="#E9F3FF", alpha=0.85, zorder=0)
    ax.axhline(threshold*100, linestyle="--", color="#666666", linewidth=1, zorder=1)
    ax.plot(x, y, color=gray, linewidth=1.8, zorder=2)
    # All observations keep their true values. Ambiguous points appear gray.
    colors = np.where(s.regime.to_numpy()==1, blue,
                      np.where(s.regime.to_numpy()==-1, red, gray))
    ax.scatter(x, y, c=colors, s=45, edgecolor="white", linewidth=0.5, zorder=3)
    ax.set_title(f"{item.example_label}  (Switches = {int(item.switches)})", fontsize=12, fontweight="bold")
    ax.set_ylim(0, max_display)
    ax.set_xlim(YEAR_START-0.3, YEAR_END+0.3)
    ax.set_xticks(np.arange(YEAR_START, YEAR_END+1, 2))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.grid(axis="y", color="#DDDDDD", linewidth=0.6)
    ax.set_axisbelow(True)

fig.suptitle("Repeated Switching in Household Healthy Purchasing Behavior", fontsize=18, fontweight="bold", y=0.97)
fig.text(0.5, 0.89, "Annual healthy share of food spending (V38 five categories)", ha="center", fontsize=11)
fig.supxlabel("Calendar Year", fontsize=11, y=0.19)
fig.supylabel("Healthy Share of Food Spending (%)", fontsize=11, x=0.02)
legend = [
    Line2D([0],[0], color="none", marker="o", markerfacecolor=blue, markeredgecolor="white", markersize=8, label="Higher healthy purchasing"),
    Line2D([0],[0], color="none", marker="o", markerfacecolor=red, markeredgecolor="white", markersize=8, label="Lower healthy purchasing"),
    Line2D([0],[0], color="none", marker="o", markerfacecolor=gray, markeredgecolor="white", markersize=8, label="Near threshold"),
    Line2D([0],[0], color="#666666", linestyle="--", label=f"Pooled median = {threshold*100:.1f}%"),
]
fig.legend(handles=legend, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.11), fontsize=9)
fig.text(0.055, 0.025,
         f"Notes: Selected households have ≥{MIN_CONSECUTIVE_YEARS} consecutive years, ≥{MIN_ANNUAL_TRIPS} trips/year and ≥{MIN_SWITCHES} crossings "
         f"between regimes outside a ±{EXCLUSION_BAND_PP:g} pp band. V38 categories include fruits and vegetables, "
         "yogurt, cereal, snacks, and beverages (with ordinary milk). Descriptive regimes are not HMM latent states.",
         ha="left", va="bottom", fontsize=8, color="#444444", wrap=True)
fig.subplots_adjust(left=0.075, right=0.985, top=0.75, bottom=0.30, wspace=0.16)
for extension in ("png", "pdf"):
    fig.savefig(OUTPUT_DIR / f"repeated_switching_three_households_v38.{extension}", dpi=300, bbox_inches="tight")
plt.show()

settings = {
    "category_specification": ["Fruits and Vegetables", "Yogurt", "Cereal", "Snacks", "Beverages including milk"],
    "beverage_merging_rule": "milk_y > 0 or beverage_y == 1 yields healthy-choice code 1",
    "healthy_milk_components": HEALTHY_COMPONENTS["beverage"][1:],
    "aggregation_version": AGGREGATION_VERSION,
    "healthy_spending_columns": HEALTHY_SPENDING_COLUMNS,
    "denominator": "sum of annual trip food_spend",
    "threshold_definition": "Pooled median among valid household-years",
    "median_threshold": threshold,
    "exclusion_band_pp": EXCLUSION_BAND_PP,
    "min_switches": MIN_SWITCHES,
    "min_consecutive_years": MIN_CONSECUTIVE_YEARS,
    "min_annual_trips": MIN_ANNUAL_TRIPS,
    "eligible_households": len(eligible),
    "selection": "Purposive: prioritize switch count, then range, spell length, seeded tie-break",
}
(OUTPUT_DIR / "figure_settings_v38.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print(f"Private figure and audit outputs: {OUTPUT_DIR}")


## 8. V38 output inventory and interpretation

The output directory is **private** and separate from GitHub:
`healthy_purchase_data/model_outputs/model_free_evidence/v38_plain_milk_beverage/switching_and_variance/`

- `annual_v38_five_category_spending.parquet`, `.meta.json`: validated household-year cache, not publicly shareable.
- `input_audit.csv`, `v38_beverage_merge_audit.csv`: source counts and V38 milk/beverage choice-merging cross-tabulation.
- `variance_sample_flow.csv`: household-year retention under the original 12-trip minimum.
- `variance_decomposition_v38_five_category.csv`, `.png`, `.pdf`: exact between/within SS, component variance, and shares, plus Figure A.
- `variance_decomposition_v38_long_panel_robustness.csv`: sensitivity selecting households having a ≥8-year valid spell (if available).
- `yearly_cross_sectional_variance_v38_five_category.csv`: each year's cross-household moments.
- `annual_cross_sectional_variance_v38_five_category.png`, `.pdf`: yearly cross-household variance (Figure B).
- `screening_counts_v38.csv`: selection audit for three example households.
- `selected_households_v38_private.csv` and `selected_trajectories_v38_private.parquet`: identifiers and trajectories; **never upload to public GitHub**.
- `repeated_switching_three_households_v38.png`, `.pdf`: original-style Figure C with anonymized labels (still review before sharing).
- `figure_settings_v38.json`: reproducible threshold, component selection and visualization settings.

**Key methodological distinction:** Between/within variance decomposition and three-household threshold crossings are **model-free descriptive evidence**, not estimated transition probabilities from the HMM. V38 is a **dummy choice-emission specification**; the annual spending share here is an intentionally separate model-free measurement aligned to V38's categories. The amount of cross-sectional/within variation may change relative to V30 or four-category results simply because the numerator definition and eligible sample differ.
